# Web Scraping — Falcon 9 Launch Records

This notebook demonstrates HTML retrieval and table extraction for the historical Falcon 9 launch list. The source page may change; record the retrieval date when running and compare extracted fields with the fixed IBM course sample before using data for modelling.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import requests
from bs4 import BeautifulSoup
import pandas as pd

URL = 'https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches'
OUT = Path('outputs/web_scraping')
OUT.mkdir(parents=True,exist_ok=True)
headers = {'User-Agent':'Mozilla/5.0 (compatible; IBM-Capstone-Research/1.0)'}
response = requests.get(URL,headers=headers,timeout=30)
response.raise_for_status()
print('HTTP status:',response.status_code)
print('Retrieved UTC:',datetime.now(timezone.utc).isoformat())
html_text = response.text
(OUT/'falcon9_launches_page.html').write_text(html_text,encoding='utf-8')

## 1. Parse the page and inspect tables

BeautifulSoup parses the HTML and identifies table elements. Table structure should be inspected before selecting a table; table order can change as Wikipedia editors update the page.

In [ ]:
soup = BeautifulSoup(html_text,'html.parser')
tables = soup.find_all('table')
print('HTML tables found:',len(tables))
for i,table in enumerate(tables[:12]):
    headers = [th.get_text(' ',strip=True) for th in table.find_all('th')[:8]]
    print(i,headers)

## 2. Extract the launch table

Select the table only after confirming its header names. Pandas read_html is used after the HTML parsing step as a convenient tabular extractor. The code searches for a table containing launch date, launch site, and payload fields, rather than relying blindly on a hard-coded table index.

In [ ]:
all_tables = pd.read_html(response.text)
def norm(x): return ''.join(ch.lower() for ch in str(x) if ch.isalnum())
candidates=[]
for idx,table in enumerate(all_tables):
    cols=[norm(c) for c in table.columns]
    flat=' '.join(cols)
    if 'date' in flat and ('launchsite' in flat or 'site' in flat) and 'payload' in flat:
        candidates.append((idx,table))
if not candidates:
    raise ValueError('Could not identify a launch table with Date, Launch Site and Payload fields. Inspect all_tables and update the selection.')
table_index,launch_table=candidates[0]
print('Selected table index:',table_index)
print('Extracted shape:',launch_table.shape)
display(launch_table.head(10))

## 3. Clean extracted text and export

Normalize multi-row headers where needed, remove footnote markers and whitespace, and retain the source columns needed for the data-wrangling stage. Do not claim this live table is identical to the IBM fixed lab data unless verified.

In [ ]:
clean = launch_table.copy()
clean.columns = [' '.join(str(c).replace('Unnamed:','').split()).strip() for c in clean.columns]
for c in clean.select_dtypes(include='object').columns:
    clean[c] = clean[c].astype('string').str.replace(r'\\[.*?\\]','',regex=True).str.replace(r'\\s+',' ',regex=True).str.strip()
clean = clean.dropna(how='all').reset_index(drop=True)
clean.to_csv(OUT/'falcon9_web_scraped_launch_records.csv',index=False)
print('Saved rows:',len(clean))
print('Saved file:',(OUT/'falcon9_web_scraped_launch_records.csv').resolve())
display(clean.head())

## Limitations

Wikipedia is an editable secondary source. Launch-table column structures, headings, and historical notes may change. This notebook records the live retrieval date and preserves the raw HTML. Use the exact dataset distributed with the IBM course when reproducing the course analysis.